<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap2_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

chap 2 transformer



2.1.1 テキストのトークン化

Qwen を使ってトークナイザーがどのように文書を処理するのかを見てみる

In [1]:
from transformers import AutoTokenizer

# prompt = "It was a dark and stormy"
prompt = "It was a dark and stormy night. The"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B")
input_ids = tokenizer(prompt).input_ids
input_ids

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[2132, 572, 264, 6319, 323, 13458, 88, 3729, 13, 576]

In [2]:
for t in input_ids:
  print(t, "\t:", tokenizer.decode(t))

2132 	: It
572 	:  was
264 	:  a
6319 	:  dark
323 	:  and
13458 	:  storm
88 	: y
3729 	:  night
13 	: .
576 	:  The


備考）トークナイザーの訓練はモデルの訓練とは明らかに異なる。モデルの訓練が確率的であるのに対し、トークナイザーの訓練は決定論的である。トークナイザーの訓練は与えられたトークンをどのように処理するのかを特定するためのもの。

2.1.2 確率の予測

GPT-2 や Qwen, SmolLM は因果言語モデル（自己回帰ともいう）と呼ばれる。先行したトークンが与えられた時に、シーケンスの次のトークンを予測するように訓練されていることから。

この節ではモデルがどのように予測しているのかを見てみる。

transformer ライブラリには多くのモデルとそれに対応するトークナイザーを備えているが、個別に指定する必要はない。なぜならAutoTokenizer や AutoModelForCausalLM によって処理に適したものが選定されるから。

In [3]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [4]:
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model(input_ids)
outputs.logits.shape # 各入力トークンを出力する

# torch.Size([1, 7, 151936])
# バッチ数、シーケンス長（トークン数）、語彙数（トークンごとの判定に用いられる語彙数（１５万））

torch.Size([1, 10, 151936])

ロジットとは？ Gemini より

言語モデルの文脈における「ロジット」とは、モデルが次のトークンを予測する際に出力する生の値のことです。これらの値はまだ確率に変換されていません。

具体的には、モデルが「It was a dark and stormy」という入力を受け取った後、次に続く可能性のあるすべての単語（語彙内のすべてのトークン）に対して、どれくらいその単語が適切かを示す数値を出力します。この数値がロジットです。

例えば、outputs.logits.shapeがtorch.Size([1, 7, 151936])という出力だった場合、これは以下を意味します。

1: バッチサイズ（入力が1つであるため）。
7: シーケンス長（入力トークンの数、「It was a dark and stormy」が7トークンに分解されたため）。
151936: 語彙サイズ（モデルが知っているすべての単語の数）。
つまり、各入力トークン（例: 'stormy'の前の'storm'）に対して、次に続く可能性のある151936個のすべての単語それぞれにロジット値が割り当てられます。これらのロジット値は、正負の無限大までの範囲をとることができ、高い値ほどそのトークンが次に続く可能性が高いことを示唆します。

最終的に、これらのロジットはソフトマックス関数（または同様の関数）を通過することで、0から1の間の確率値に変換され、その確率に基づいて最も可能性の高い次のトークンが選択されます。

In [5]:
final_logits = model(input_ids).logits[0, -1] # 最後のトークンに対応するロジット
final_logits.argmax() # 最大値のインデックス

# output -> tensor(3729)

tensor(12884)

In [6]:
tokenizer.decode(final_logits.argmax())

# output -> ' night'
# 次に続く単語が night である確率が最も高いということがわかる（ロジットの最大値がこれってこと）

' sky'

topk() メソッドを使って、他にどのようなトークンが候補に上がっていたかを見てみる。

In [7]:
import torch
top10_logits = torch.topk(final_logits, 10)
for index in top10_logits.indices:
  print(tokenizer.decode(index))

#  night
#  evening
#  day
#  morning
#  winter
#  afternoon
#  Saturday
#  Sunday
#  Friday
#  October

 sky
 wind
 storm
 rain
 moon
 weather
 sun
 stars
 only
 air


候補となっていたトークンがどれくらいの確率だったかをみるために、ロジットを確率に変換してみる。そのために softmax() で正規化する。

In [8]:
top10 = torch.topk(final_logits.softmax(dim=0), 10)
for value, index in zip(top10.values, top10.indices):
  print(f"{tokenizer.decode(index):<10} {value.item():.2%}")

 sky       9.40%
 wind      8.00%
 storm     4.90%
 rain      3.83%
 moon      2.28%
 weather   2.23%
 sun       2.23%
 stars     1.94%
 only      1.81%
 air       1.80%


ここまでの処理を input の文章を変えてみて色々試してみるとロジットの確率も結構変化することがわかった。

2.1.3 テキストの生成

次のトークンの確率を取得して、モデルに入力し続けることで文章の生成が可能になる。この用途に最適なメソッドが generate() である。

In [9]:
output_ids = model.generate(input_ids, max_new_tokens=20)
decoded_text = tokenizer.decode(output_ids[0])

print("Input Ids", input_ids[0])
print("Output IDs", output_ids)
print(f"Generated text : {decoded_text}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Input Ids tensor([ 2132,   572,   264,  6319,   323, 13458,    88,  3729,    13,   576])
Output IDs tensor([[ 2132,   572,   264,  6319,   323, 13458,    88,  3729,    13,   576,
         12884,   572,  6319,   323,   279,  9956,   572,  1246,  2718,    13,
           576, 11174,   572, 50413,  1495,   323,   279, 32438,   572, 49757]])
Generated text : It was a dark and stormy night. The sky was dark and the wind was howling. The rain was pouring down and the lightning was flashing


このように最も確率が高いトークンを生成する方法がグリーディデコーディング。
文全体の整合性などが考慮されない点が難点。

一つのトークンだけでなく、シーケンスの複数のパターンを加味した確率で文章を生成する方法をビームサーチ（beam search）という。↓

In [10]:
beam_output = model.generate(
    input_ids,
    num_beams=5, # ５個の仮説を保持し、この中から最も可能性の高いものを採用する
    max_new_tokens=30,
)

print(tokenizer.decode(beam_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The wind was howling, and the rain was pouring down. The sky was dark and gloomy, and the air was filled with the sound of thunder


モデルによっては繰り返しが多くなることがある。
繰り返しが少なくなるように、repetition_penalty や bad_wards_ids を指定することで回避する。

In [11]:
beam_output = model.generate(
    input_ids,
    num_beams=5,
    repetition_penalty=2.0,
    max_new_tokens=38,
)

print(tokenizer.decode(beam_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The wind was howling, the rain was pouring, and the sky was filled with thunder and lightning. I was sitting in my car, driving home from work, when all of a sudden,


人間が書く優れた文章は実は確率が高いものが選定されていない。これにならって、サンプリングという手法を採用することもできる。
サンプリングは確率が最も高いものを選ぶのではなく、その時点の確率分布から抽選して単語を採用する。
transformer では do_sample を使うことでサンプリングができる。

In [12]:
from transformers import set_seed

set_seed(70)

sampling_output = model.generate(
    input_ids,
    do_sample=True,
    max_new_tokens=34,
    top_k=0 #あとで解説
)

print(tokenizer.decode(sampling_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The snow snapped the pots of moisture. I was awed at the mental outfit my dogs established shared with me, on either side of this statute. I felt awake at attention


温度（temperature）パラメーターを使うことで、サンプリング前に確率分布を操作できる。分布の勾配を急にしたり平坦にしたりすることができる。温度が1.0より高ければ分布のランダム性が高く、逆に温度が0.0の場合はグリーディデコーディングと同様に最も確率が高いものを採用するようになる。

In [13]:
sampling_output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.4,
    max_new_tokens=40,
    top_k=0
)

print(tokenizer.decode(sampling_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The skies were dark and the wind was howling. The rain was pouring down and the wind was howling. The rain was pouring down and the wind was howling. The rain was pouring down and


In [14]:
sampling_output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.001,
    max_new_tokens=40,
    top_k=0
)

print(tokenizer.decode(sampling_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The sky was dark and the wind was howling. The rain was pouring down and the lightning was flashing. The sky was dark and the wind was howling. The rain was pouring down and the lightning


In [15]:
sampling_output = model.generate(
    input_ids,
    do_sample=True,
    temperature=3.0,
    max_new_tokens=40,
    top_k=0
)

print(tokenizer.decode(sampling_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The whe느 exhilar swords seas rolled NUM HibernateOthers Турية freed deploy Exhibition strtotimering finishing invadingmarker honoringЩ Uniform barracks Joan onde Emmapeutic/get铟recommended Cant Ant municipalities Kgforeach covering kin grown tacticalButtonText


top_k パラメーターは次に来るトークンのうち最も確率が高い上位k個を候補とするパラメーター。

In [16]:
sampling_output = model.generate(
    input_ids,
    do_sample=True,
    max_new_tokens=40,
    top_k=5
)

print(tokenizer.decode(sampling_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The rain was coming down hard, and the wind was blowing like a hurricane. I was walking through the forest, trying to find a place to hide, when I heard a noise. My heart raced as


top_p パラメーターは累積確率が所定の値を超えるまで次に来るトークンを候補とする方法。top_k だと低い確率のトークン候補も対象に含まれてしまうことがあることから、top_p を用いる手法がある。

In [17]:
sampling_output = model.generate(
    input_ids,
    do_sample=True,
    max_new_tokens=40,
    top_k=0,
    top_p=0.94
)

print(tokenizer.decode(sampling_output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


It was a dark and stormy night. The lights in the room went out, and then they flickered on. Family spent the night of May 2006, that just happened to be the night it was perceived that 9/


top_kとtop_pを組み合わせて使うこともよくある。

2.1.4 ゼロショット汎化

モデルには一切ラベル付けデータを与えずに分類をさせてみる。
定番の映画のレビューがポジティブかネガティブかのセンチメント分類をさせる。

センチメント分析の方法にはさまざまあるが、今回は positive と negative の確率が高い方を採用する方式で試してみる。

In [18]:
# 単語 positive と negative の ID を確認する
tokenizer.encode(" positive"), tokenizer.encode(" negative")

#outputs: ([6785], [8225])

([6785], [8225])

In [19]:
def score(review):
  """レビューがポジティブかネガティヴかを予測する。
    少しプロンプトを工夫して、それぞれのトークンに対応するロジットを参照し、よりスコアの高いラベルを返す。
  """
  prompt = f"""Question: Is the following review positive or negative about the movie? Review: {review} Answer: """
  input_ids = tokenizer(prompt, return_tensors="pt").input_ids
  final_logits = model(input_ids).logits[0, -1]
  if final_logits[6785] > final_logits[8225]:
    print("Positive")
  else:
    print("Negative")

In [20]:
score("This movie was terrible!")

Negative


In [21]:
score("That movie was great!")

Positive


In [22]:
score("A complex yet wonderful film about the depravity of man")

Positive


2.1.5 少数ショット汎化

In [23]:
prompt = """\
Translate English to Spanish:

English: I do not speak Spanish.
Spanish: No hablo espanol.

English: See you later!
Spanish: i Hasta luego!

English: Where is a good restaurant?
Spanish: ?Donde hay un buen restraurante?

English: What rooms do you have avaiable?
Spanish: ?Que habitaciones tiene disponible?

English: I like soccer.
Spanish: """
inputs = tokenizer(prompt, return_tensors="pt").input_ids
output = model.generate(
    inputs,
    max_new_tokens=10,
)

print(tokenizer.decode(output[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Translate English to Spanish:

English: I do not speak Spanish.
Spanish: No hablo espanol.

English: See you later!
Spanish: i Hasta luego!

English: Where is a good restaurant?
Spanish: ?Donde hay un buen restraurante?

English: What rooms do you have avaiable?
Spanish: ?Que habitaciones tiene disponible?

English: I like soccer.
Spanish: 1. ¿Qué piensas de la soccer


2.2 Transformer ブロック

・トークン化
・入力トークン埋め込み
・位置エンコーディング :Transformer 自体には語順の概念がないので、トークンの埋め込みに位置情報を加える
・Transformer ブロック
・文脈埋め込み
・予測

2.3 Transformer モデルの系譜

2.3.1 Seq2Seq タスク: エンコーダー・デコーダー型アーキテクチャー

2.3.2 エンコーダーのみのモデル

かつての GPT-2.0 の時代ではデコーダーのみのモデルは予測タスクに適しており、翻訳などのタスクはエンコーダー付きの Sec2Sec のほうが適していた（エンコーダーデコーダー型は入力シーケンスを出力シーケンスに変換する設計であるため）。
しかし、昨今の GPT-4.0 以降のモデルの性能向上により、翻訳タスクについてもデコーダーのみのモデルで対応できるようになった。

これまで使った AutoModel や AutoTokenizer クラスを使わずに、特定のタスク向けのモデルを手軽に読み込める transformers の API、pipeline を使ってみる。

In [24]:
from transformers import pipeline

fill_masker = pipeline("fill-mask", model="bert-base-uncased")
fill_masker("The [MASK] is made of milk.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cpu


[{'score': 0.19546712934970856,
  'token': 9841,
  'token_str': 'dish',
  'sequence': 'the dish is made of milk.'},
 {'score': 0.12907519936561584,
  'token': 8808,
  'token_str': 'cheese',
  'sequence': 'the cheese is made of milk.'},
 {'score': 0.1059068813920021,
  'token': 6501,
  'token_str': 'milk',
  'sequence': 'the milk is made of milk.'},
 {'score': 0.04112080857157707,
  'token': 4392,
  'token_str': 'drink',
  'sequence': 'the drink is made of milk.'},
 {'score': 0.03712376952171326,
  'token': 7852,
  'token_str': 'bread',
  'sequence': 'the bread is made of milk.'}]

2.4 事前訓練済みモデルの威力

訓練済みのベースモデルをもとにファインチューニング（転移学習）したモデルを利用すれば、学習コストを大幅に小さくすることができる。

テキストのセンチメント分類向けにファインチューニングされたモデルを使ってみる。

In [26]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
classifier("This movie is disgustingly good!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9998536109924316}]

2.5 Transformer の総括